## **Cadenas de Markov por producto: Alertas de antigüedad**

Estima, para cada SKU, la probabilidad de pasar de un estado de alerta de antigüedad en el mes *t* a otro en el mes *t + 1*.

**Estados**

| Estado | Regla (igual que `fact_kardex` en Power BI) |
|---|---|
| Verde | días de antigüedad ≤ `dias_verde` |
| Ámbar | `dias_verde` < días ≤ `dias_rojo` |
| Rojo | días > `dias_rojo` |
| FULCOM futura | FULCOM > periodo |
| Sin fecha | FULCOM nula |
| Baja | STOCK ≤ 0, o el SKU deja de aparecer en el mes siguiente |

Los días se calculan como `periodo − FULCOM`, donde `periodo` es el fin de mes sacado del nombre del archivo `kardex_YYYYMM.xlsx`. Antes de asignar estados se corrige la FULCOM cuando un corte trae el año equivocado (sección 3).

**Salida:** `data/processed/markov_transiciones_sku.csv`, con una fila por SKU × estado origen × estado destino.

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Configuración

In [ ]:
# Rutas (se pueden cambiar con variables de entorno)
KARDEX_DIR = Path(os.environ.get('KARDEX_DIR', '../data/raw'))
PARAMETROS_XLSX = Path(os.environ.get('PARAMETROS_XLSX', '../templates/Parametros.xlsx'))
SALIDA_CSV = Path(os.environ.get('SALIDA_CSV', '../data/processed/markov_transiciones_sku.csv'))

# Peso del prior global en el suavizado Dirichlet (equivale a k transiciones "ficticias")
K_SUAVIZADO = 1.0

COLUMNAS_KARDEX = ['PERIODO', 'CODIGO', 'DESCRIPTOR', 'STOCK', 'COSTO', 'VALORIZACION', 'FULCOM',
                   'UNID CONV', 'LINEA', 'NOMBRE LINEA', 'POLITICA CJE AL 30SEP']
COLUMNAS_TEXTO = ['CODIGO', 'DESCRIPTOR', 'LINEA', 'NOMBRE LINEA', 'POLITICA CJE AL 30SEP']

# Acepta kardex_YYYYMM.xlsx (DVC) e INV_Kardex_YYYYMM.xlsx (originales de SharePoint)
PATRON_ARCHIVO = re.compile(r'(?:INV_)?kardex_(\d{6})\.xlsx', re.IGNORECASE)

ESTADOS = ['Verde', 'Ámbar', 'Rojo', 'FULCOM futura', 'Sin fecha', 'Baja']

### 1. Parámetros

Se leen de `Parametros.xlsx`, la misma tabla que consume Power BI, para no tener umbrales fijos en el código.

In [ ]:
parametros = pd.read_excel(PARAMETROS_XLSX)
parametros = parametros.set_index('parametro')['valor']

DIAS_VERDE = float(parametros['dias_verde'])
DIAS_ROJO = float(parametros['dias_rojo'])

print(f'dias_verde = {DIAS_VERDE:g} | dias_rojo = {DIAS_ROJO:g}')

### 2. Carga de todos los meses

Se leen todos los `kardex_YYYYMM.xlsx` o `INV_Kardex_YYYYMM.xlsx` de `KARDEX_DIR`. Se usa la hoja `DETALLE KARDEX` si existe; si no, la primera hoja.

In [ ]:
def leer_kardex(ruta: Path) -> pd.DataFrame:
    libro = pd.ExcelFile(ruta)
    hojas = {h.strip().upper(): h for h in libro.sheet_names}
    hoja = hojas.get('DETALLE KARDEX', libro.sheet_names[0])
    df = pd.read_excel(libro, sheet_name=hoja)
    df.columns = [str(c).strip() for c in df.columns]
    df = df[COLUMNAS_KARDEX].copy()

    # Periodo = fin de mes según el nombre del archivo (igual que Kardex_Origen)
    yyyymm = PATRON_ARCHIVO.fullmatch(ruta.name).group(1)
    df['periodo'] = pd.Timestamp(f'{yyyymm[:4]}-{yyyymm[4:]}-01') + pd.offsets.MonthEnd(0)
    df['archivo'] = ruta.name
    return df


archivos = sorted((p for p in KARDEX_DIR.glob('*.xlsx') if PATRON_ARCHIVO.fullmatch(p.name)),
                  key=lambda p: PATRON_ARCHIVO.fullmatch(p.name).group(1))
assert archivos, f'No se encontraron archivos kardex_YYYYMM.xlsx / INV_Kardex_YYYYMM.xlsx en {KARDEX_DIR}'

data = pd.concat([leer_kardex(p) for p in archivos], ignore_index=True)

# Limpieza (equivalente a Kardex_Limpio)
for col in COLUMNAS_TEXTO:
    data[col] = data[col].astype('string').str.strip().replace('', pd.NA)
for col in ['STOCK', 'COSTO', 'VALORIZACION', 'UNID CONV']:
    data[col] = pd.to_numeric(data[col], errors='coerce')
data['FULCOM'] = pd.to_datetime(data['FULCOM'], errors='coerce')

# Filas sin código no pertenecen a ningún SKU
data = data[data['CODIGO'].notna()].copy()

data.groupby('periodo').agg(archivo=('archivo', 'first'), filas=('CODIGO', 'size'),
                            skus=('CODIGO', 'nunique'), valorizacion=('VALORIZACION', 'sum'))

In [ ]:
PERIODOS = sorted(data['periodo'].unique())
print(f'Periodos cargados: {len(PERIODOS)}')

if len(PERIODOS) < 2:
    raise ValueError('Se necesitan al menos 2 meses de Kardex para observar transiciones. '
                     f'Solo se encontró: {[str(p.date()) for p in PERIODOS]}')

# Duplicados (mismo SKU en el mismo periodo): se conserva el de mayor valorización
duplicados = data.duplicated(['periodo', 'CODIGO'], keep=False)
print(f'Filas con SKU duplicado en el mismo periodo: {duplicados.sum()}')
data = (data.sort_values('VALORIZACION', ascending=False)
            .drop_duplicates(['periodo', 'CODIGO'])
            .sort_values(['CODIGO', 'periodo'])
            .reset_index(drop=True))

### 3. Corrección de FULCOM con año equivocado

`FULCOM` es la fecha de la última compra. Para un mismo SKU **no puede retroceder** de un corte al siguiente: una compra nueva la hace avanzar y, si no hay compra, se mantiene.

En los datos aparece un error con una firma muy clara: en un corte, la FULCOM de algunos SKU es **exactamente un año menor** (mismo día y mes) que la del corte anterior, y el corte siguiente arrastra el valor erróneo si no hubo compra nueva. Esa fecha queda antes que FULCOM que el mismo SKU ya tenía en meses anteriores, así que es imposible y no puede ser una compra real.

**Regla.** Se recorren los cortes en orden. Si la FULCOM de un SKU en *t* + 1 año es igual a su última FULCOM válida (ya corregida), se le repone el año. Se usa la última válida, y no solo la de *t − 1*, para que el arrastre del error también se corrija.

- No se toca ninguna otra fecha. Un salto **hacia adelante** de un año es una compra nueva y es legítimo.
- Cualquier otro retroceso se reporta como `retrocesos_residuales` y no se corrige, porque no se puede saber cuál es la fecha correcta.
- `FULCOM_original` conserva el valor del archivo y `fulcom_corregida` marca cada fila corregida.

In [ ]:
data['FULCOM_original'] = data['FULCOM']
data['fulcom_corregida'] = False

ultima_valida = pd.Series(dtype=data['FULCOM'].dtype)   # última FULCOM no nula (ya corregida) por SKU
for periodo in PERIODOS:
    filas = data.index[data['periodo'] == periodo]
    actual = data.loc[filas, 'FULCOM']
    previa = pd.Series(ultima_valida.reindex(data.loc[filas, 'CODIGO']).to_numpy(), index=filas)

    error_anio = actual.notna() & previa.notna() & (actual + pd.DateOffset(years=1) == previa)
    corregir = error_anio[error_anio].index
    data.loc[corregir, 'FULCOM'] = previa[corregir]
    data.loc[corregir, 'fulcom_corregida'] = True

    vigente = data.loc[filas].dropna(subset=['FULCOM']).set_index('CODIGO')['FULCOM']
    ultima_valida = vigente.combine_first(ultima_valida)

# Retrocesos que quedan. Solo los de meses consecutivos entran en la cadena (sección 5);
# los que ocurren tras una ausencia del SKU no forman transición y solo se informan.
g = data.groupby('CODIGO')
fulcom_previa, periodo_previo = g['FULCOM'].shift(), g['periodo'].shift()
consecutivo = periodo_previo + pd.offsets.MonthEnd(1) == data['periodo']
retroceso = data['FULCOM'].notna() & fulcom_previa.notna() & (data['FULCOM'] < fulcom_previa)
retrocesos_residuales = int((retroceso & consecutivo).sum())

ultima_no_nula = g['FULCOM'].transform(lambda s: s.ffill().shift())
retrocesos_tras_ausencia = int((data['FULCOM'].notna() & ultima_no_nula.notna()
                                & (data['FULCOM'] < ultima_no_nula) & ~(retroceso & consecutivo)).sum())

print(f'Filas con FULCOM corregida: {data["fulcom_corregida"].sum():,} '
      f'| SKU afectados: {data.loc[data["fulcom_corregida"], "CODIGO"].nunique():,}')
print(f'Retrocesos sin explicar entre meses consecutivos (entran en la cadena): {retrocesos_residuales:,}')
print(f'Retrocesos tras una ausencia del SKU (no forman transición): {retrocesos_tras_ausencia:,}')
if retrocesos_residuales:
    print('ATENCIÓN: hay retrocesos de FULCOM distintos al error de año; revisar antes de usar la cadena.')

data.groupby(data['periodo'].dt.strftime('%Y-%m'))['fulcom_corregida'].sum().rename('filas_corregidas').to_frame()

### 4. Estado de cada SKU en cada mes

In [ ]:
dias = (data['periodo'] - data['FULCOM']).dt.days

data['estado'] = np.select(
    [data['STOCK'].isna() | (data['STOCK'] <= 0),
     data['FULCOM'].isna(),
     data['FULCOM'] > data['periodo'],
     dias <= DIAS_VERDE,
     dias <= DIAS_ROJO],
    ['Baja', 'Sin fecha', 'FULCOM futura', 'Verde', 'Ámbar'],
    default='Rojo')

pd.crosstab(data['estado'], data['periodo'].dt.strftime('%Y-%m')).reindex(ESTADOS, fill_value=0)

### 5. Panel de transiciones *t → t + 1*

Solo se emparejan meses **consecutivos** que se hayan cargado. Si falta un archivo intermedio, ese salto no se cuenta. Si un SKU existe en *t* y no aparece en *t + 1*, su destino es `Baja`.

In [ ]:
siguiente_mes = {p: p + pd.offsets.MonthEnd(1) for p in PERIODOS}
periodos_set = set(PERIODOS)

origen = data[['CODIGO', 'periodo', 'estado']].copy()
origen['periodo_destino'] = origen['periodo'].map(siguiente_mes)
origen = origen[origen['periodo_destino'].isin(periodos_set)]   # solo meses consecutivos disponibles

destino = data[['CODIGO', 'periodo', 'estado']].rename(columns={'periodo': 'periodo_destino',
                                                                 'estado': 'estado_destino'})

pares = (origen.rename(columns={'estado': 'estado_origen'})
               .merge(destino, on=['CODIGO', 'periodo_destino'], how='left'))
pares['estado_destino'] = pares['estado_destino'].fillna('Baja')   # desapareció en t + 1

print(f'Transiciones observadas: {len(pares):,} | SKU con al menos una: {pares["CODIGO"].nunique():,}')
pares.head()

### 6. Matriz de transición global

Se estima por máxima verosimilitud con todos los SKU, y sirve de referencia y de *prior* para las matrices por producto. Si un estado de origen nunca se observa, su fila se deja uniforme.

In [ ]:
conteo_global = (pd.crosstab(pares['estado_origen'], pares['estado_destino'])
                   .reindex(index=ESTADOS, columns=ESTADOS, fill_value=0))

filas_vacias = conteo_global.sum(axis=1) == 0
P_global = conteo_global.div(conteo_global.sum(axis=1).replace(0, np.nan), axis=0)
P_global.loc[filas_vacias] = 1 / len(ESTADOS)

if filas_vacias.any():
    print('Estados de origen sin observaciones (fila uniforme):', list(P_global.index[filas_vacias]))

assert np.allclose(P_global.sum(axis=1), 1)
P_global.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(P_global, annot=True, fmt='.2f', cmap='Reds', vmin=0, vmax=1, cbar_kws={'label': 'Probabilidad'}, ax=ax)
ax.set_xlabel('Estado en t + 1')
ax.set_ylabel('Estado en t')
ax.set_title('Matriz de transición global de alertas de antigüedad')
plt.tight_layout()
plt.show()

### 7. Matriz de transición por producto (suavizada)

Cada SKU tiene como mucho *(meses − 1)* transiciones, así que la estimación directa (MLE) es muy ruidosa. Por eso se aplica un *prior* Dirichlet centrado en la matriz global:

$$P_{sku}(i \to j) = \frac{n_{ij} + k \cdot P_{global}(i \to j)}{n_i + k}$$

- Con pocas observaciones, la fila se acerca a la global; con muchas, domina la historia propia del SKU.
- Si el SKU nunca estuvo en el estado *i*, la fila es exactamente la global.
- `prob_mle` queda vacía cuando $n_i = 0$.

In [ ]:
skus = pd.Index(sorted(data['CODIGO'].unique()), name='CODIGO')

malla = pd.MultiIndex.from_product([skus, ESTADOS, ESTADOS], names=['CODIGO', 'estado_origen', 'estado_destino'])
transiciones = (pares.groupby(['CODIGO', 'estado_origen', 'estado_destino']).size()
                     .reindex(malla, fill_value=0).rename('conteo').reset_index())

transiciones['n_origen'] = transiciones.groupby(['CODIGO', 'estado_origen'])['conteo'].transform('sum')

prior = P_global.stack().rename('prob_global').reset_index()
prior.columns = ['estado_origen', 'estado_destino', 'prob_global']
transiciones = transiciones.merge(prior, on=['estado_origen', 'estado_destino'], how='left')

transiciones['prob_mle'] = transiciones['conteo'] / transiciones['n_origen'].replace(0, np.nan)
transiciones['prob_suavizada'] = ((transiciones['conteo'] + K_SUAVIZADO * transiciones['prob_global'])
                                  / (transiciones['n_origen'] + K_SUAVIZADO))

transiciones.head(12)

### 8. Validaciones

In [ ]:
# Cada fila (SKU, estado_origen) suma 1
suma_suav = transiciones.groupby(['CODIGO', 'estado_origen'])['prob_suavizada'].sum()
assert np.allclose(suma_suav, 1), 'Hay filas suavizadas que no suman 1'

suma_mle = transiciones.dropna(subset=['prob_mle']).groupby(['CODIGO', 'estado_origen'])['prob_mle'].sum()
assert np.allclose(suma_mle, 1), 'Hay filas MLE que no suman 1'

# Los conteos coinciden con los pares consecutivos observados
assert transiciones['conteo'].sum() == len(pares)
assert (conteo_global.values.sum()) == len(pares)

# Una fila por SKU × origen × destino
assert len(transiciones) == len(skus) * len(ESTADOS) ** 2

# Con FULCOM que no retrocede y pasos de un mes (≤ 31 días), la antigüedad sube como mucho 31 días por paso.
# Si dias_rojo − dias_verde ≥ 31, pasar de Verde a Rojo en un mes es imposible.
max_dias_paso = max((b - a).days for a, b in zip(PERIODOS, PERIODOS[1:]) if b == a + pd.offsets.MonthEnd(1))
if retrocesos_residuales == 0 and DIAS_ROJO - DIAS_VERDE >= max_dias_paso:
    assert conteo_global.loc['Verde', 'Rojo'] == 0, 'Hay transiciones Verde → Rojo imposibles'

print('Validaciones OK')

In [ ]:
def matriz_sku(codigo: str, columna: str = 'prob_suavizada') -> pd.DataFrame:
    m = transiciones[transiciones['CODIGO'] == codigo]
    return m.pivot(index='estado_origen', columns='estado_destino', values=columna).reindex(index=ESTADOS, columns=ESTADOS)


# SKU de ejemplo: uno del top 10 Rojo del EDA (si existe) y el de más transiciones
ejemplos = [c for c in ['SKU_006155'] if c in skus]
ejemplos.append(pares['CODIGO'].value_counts().idxmax())

for codigo in dict.fromkeys(ejemplos):
    print(f'\n===== {codigo} =====')
    print('Historia:', ' → '.join(data.loc[data['CODIGO'] == codigo, 'estado']))
    display(matriz_sku(codigo, 'conteo').astype(int))
    display(matriz_sku(codigo).round(3))

### 9. Exportar para Power BI

Formato largo: una fila por SKU × estado origen × estado destino, con atributos del producto tomados de su último registro.

In [ ]:
ultimo = data.sort_values('periodo').groupby('CODIGO').tail(1).set_index('CODIGO')
ultimo_periodo = max(PERIODOS)

atributos = pd.DataFrame({
    'DESCRIPTOR': ultimo['DESCRIPTOR'],
    'LINEA': ultimo['LINEA'],
    'NOMBRE LINEA': ultimo['NOMBRE LINEA'],
    'POLITICA CJE AL 30SEP': ultimo['POLITICA CJE AL 30SEP'],
    # Si el SKU no aparece en el último mes cargado, su estado actual es Baja
    'estado_actual': np.where(ultimo['periodo'] == ultimo_periodo, ultimo['estado'], 'Baja'),
    'valorizacion_actual': np.where(ultimo['periodo'] == ultimo_periodo, ultimo['VALORIZACION'], 0.0),
    'n_meses_observados': data.groupby('CODIGO')['periodo'].nunique(),
    'periodo_inicio': data.groupby('CODIGO')['periodo'].min().dt.date,
    'periodo_fin': ultimo['periodo'].dt.date,
})

salida = transiciones.merge(atributos, left_on='CODIGO', right_index=True, how='left')
salida = salida[['CODIGO', 'DESCRIPTOR', 'LINEA', 'NOMBRE LINEA', 'POLITICA CJE AL 30SEP',
                 'estado_origen', 'estado_destino', 'conteo', 'n_origen', 'prob_mle', 'prob_suavizada',
                 'estado_actual', 'valorizacion_actual', 'n_meses_observados', 'periodo_inicio', 'periodo_fin']]
salida[['prob_mle', 'prob_suavizada']] = salida[['prob_mle', 'prob_suavizada']].round(6)

SALIDA_CSV.parent.mkdir(parents=True, exist_ok=True)
salida.to_csv(SALIDA_CSV, index=False, encoding='utf-8-sig')

print(f'{SALIDA_CSV} -> {len(salida):,} filas | {salida["CODIGO"].nunique():,} SKU')
salida.head()